In [ ]:
#Cell 1 –Configuration
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path, PureWindowsPath
from typing import Dict, List, Optional, Tuple

#USER CONFIG ──
DATASET_ROOT       = Path("/kaggle/input/your-dataset")# folder with .wav files
ANNOTATIONS_JSON   = Path("/kaggle/input/your-dataset/annotations.json")# Whombat export

# Spectrogram rendering
SR_TARGET: Optional[int] = None# None= native sample rate
N_FFT        = 2048
HOP_LENGTH   = 512
N_MELS       = 128
F_MIN        = 0
F_MAX: Optional[int] = None# None = sr / 2
CMAP         = "magma"

# Visualisation
MAX_ANNOTATIONS_TO_SHOW = 30
BBOX_COLOR     = "cyan"
BBOX_LINEWIDTH = 1.5
FIGURE_DPI     = 120

print("Config loaded ")

In [ ]:
#Cell 2 –Whombat JSON Parser (full UUID chain resolution)
import json
import numpy as np
from dataclasses import dataclass, field


@dataclass
class ParsedAnnotation:
    """One resolved annotation with all references denormalized."""
    recording_uuid:   str
    audio_basename:   str
    audio_path:       Optional[Path]
    start_s:          float
    end_s:            float
    low_hz:           float
    high_hz:          float
    label:            str
    sound_event_uuid: str
    annotation_index: int = 0


def _basename_from_any_path(path_str: str) -> str:
    """Extract filename from a Windows or POSIX path string."""
    if "\\" in path_str or (len(path_str) >= 2 and path_str[1] == ":"):
        return PureWindowsPath(path_str).name
    return Path(path_str).name


def _resolve_audio(basename: str, search_dirs: List[Path]) -> Optional[Path]:
    """Find an audio file by basename across search directories."""
    stem = Path(basename).stem
    for d in search_dirs:
        if not d.exists():
            continue
        candidate = d / basename
        if candidate.exists():
            return candidate
        for ext in (".wav", ".WAV", ".flac", ".mp3", ".m4a"):
            candidate =d / (stem + ext)
            if candidate.exists():
                return candidate
        for f in d.rglob(f"{stem}.*"):
            if f.suffix.lower() in {".wav", ".flac", ".mp3", ".m4a"}:
                return f
    return None


def parse_whombat_json(
    json_path: Path,
    audio_search_dirs: List[Path],
    tag_key: str = "Species",
) -> List[ParsedAnnotation]:
    """
    Parse a Whombat project export JSON and return a flat list of
    fully-resolved ParsedAnnotation objects.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    data = doc.get("data") if isinstance(doc, dict) else None
    if not isinstance(data, dict):
        raise ValueError(f"Unexpected schema in {json_path}")

    # tag_id -> label
    tag_id_to_label: Dict[int, str] = {}
    for tag in data.get("tags", []) or []:
        if tag.get("key") == tag_key:
            try:
                tag_id_to_label[int(tag["id"])] = str(tag.get("value", "")).strip()
            except (KeyError, ValueError):
                continue

    # recording UUID -> (path_str, basename)
    rec_uuid_to_info: Dict[str, Tuple[str, str]] = {}
    for rec in data.get("recordings", []) or []:
        uuid = rec.get("uuid")
        path_str = rec.get("path", "")
        if uuid and path_str:
            rec_uuid_to_info[str(uuid)] = (path_str, _basename_from_any_path(path_str))

    # sound_event UUID -> (rec_uuid, start, end, low, high)
    se_uuid_to_geom: Dict[str, Tuple[str, float, float, float, float]] = {}
    for se in data.get("sound_events", []) or []:
        se_uuid = se.get("uuid")
        rec_uuid = se.get("recording")
        geom = se.get("geometry") or {}
        coords = geom.get("coordinates") if isinstance(geom, dict) else None
        if not se_uuid or not rec_uuid or not isinstance(coords, list) or len(coords) < 4:
            continue
        try:
            start_s, low_hz, end_s, high_hz = (
                float(coords[0]), float(coords[1]), float(coords[2]), float(coords[3])
            )
        except (ValueError, TypeError):
            continue
        se_uuid_to_geom[str(se_uuid)] = (str(rec_uuid), start_s, end_s, low_hz, high_hz)

    # sound_event UUID ->list[tag_id]
    se_uuid_to_tags: Dict[str, List[int]] = {}
    for ann in data.get("sound_event_annotations", []) or []:
        se_uuid =ann.get("sound_event")
        tag_ids= ann.get("tags")
        if not se_uuid or not isinstance(tag_ids, list):
            continue
        cleaned = []
        for tid in tag_ids:
            try:
                cleaned.append(int(tid))
            except (ValueError, TypeError):
                continue
        if cleaned:
            se_uuid_to_tags[str(se_uuid)] = cleaned

    #Resolve into flat list
    results: List[ParsedAnnotation] = []
    idx = 0
    for se_uuid, tag_ids in se_uuid_to_tags.items():
        if se_uuid not in se_uuid_to_geom:
            continue
        rec_uuid, start_s, end_s, low_hz, high_hz = se_uuid_to_geom[se_uuid]

        label ="unknown"
        for tid in tag_ids:
            if tid in tag_id_to_label:
                label = tag_id_to_label[tid]
                break

        audio_basename ="unknown.wav"
        if rec_uuid in rec_uuid_to_info:
            _, audio_basename = rec_uuid_to_info[rec_uuid]

        audio_path = _resolve_audio(audio_basename, audio_search_dirs)

        results.append(ParsedAnnotation(
            recording_uuid=rec_uuid,
            audio_basename=audio_basename,
            audio_path=audio_path,
            start_s=start_s, end_s=end_s,
            low_hz=low_hz, high_hz=high_hz,
            label=label,
            sound_event_uuid=se_uuid,
            annotation_index=idx,
        ))
        idx+=1

    print(f"Parsed {len(results)} annotations from {len(rec_uuid_to_info)} recordings")
    print(f"  Tags found: {tag_id_to_label}")
    print(f"  Sound events with geometry: {len(se_uuid_to_geom)}")
    return results


#Parse ──
audio_search_dirs = [DATASET_ROOT] + [
    p for p in DATASET_ROOT.iterdir() if p.is_dir()
] if DATASET_ROOT.exists() else [DATASET_ROOT]

annotations = parse_whombat_json(
    ANNOTATIONS_JSON,
    audio_search_dirs=audio_search_dirs,
    tag_key="Species",
)

resolved = sum(1 for a in annotations if a.audio_path is not None)
print(f"\nResolved audio for {resolved}/{len(annotations)} annotations")
if annotations:
    labels = set(a.label for a in annotations)
    print(f"Species: {labels}")

In [ ]:
#Cell 3 – Render mel-spectrograms with dynamic windowing & bounding boxes
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Audio cache to avoid re-reading the same file
_audio_cache: Dict[str, Tuple[np.ndarray, int]] = {}


def _load_audio_cached(path: Path) -> Tuple[np.ndarray, int]:
    key = str(path)
    if key not in _audio_cache:
        y, sr = librosa.load(str(path), sr=SR_TARGET, mono=True)
        _audio_cache[key] = (y, sr)
    return _audio_cache[key]


def compute_dynamic_window(
    start_s: float, end_s: float, audio_duration_s: float,
) -> Tuple[float, float]:
    """
    Compute a display window that centres the annotation.
    Short clips (< 1 s) -> ~3× duration or 0.9 s min window.
    Longer clips -> ~2× duration or 5.0 s min window.
    """
    clip_dur = end_s - start_s
    if clip_dur < 1.0:
        window_dur = max(0.9, clip_dur * 3.0)
    else:
        window_dur = max(5.0, clip_dur * 2.0)

    centre = (start_s + end_s) / 2.0
    win_start = centre - window_dur / 2.0
    win_end   = centre + window_dur / 2.0

    # clamp to audio bounds
    if win_start < 0:
        win_end = min(audio_duration_s, win_end - win_start)
        win_start = 0.0
    if win_end > audio_duration_s:
        win_start = max(0.0, win_start - (win_end - audio_duration_s))
        win_end = audio_duration_s

    return float(win_start), float(win_end)


def render_annotation(
    ann: ParsedAnnotation,
    ax: Optional[plt.Axes] = None,
    show_bbox: bool = True,
) -> Optional[plt.Figure]:
    """Render one annotation as a mel-spectrogram with optional bounding box."""
    if ann.audio_path is None:
        return None

    y, sr = _load_audio_cached(ann.audio_path)
    audio_dur = len(y) / sr
    f_max_eff= F_MAX if F_MAX else sr // 2

    win_start, win_end = compute_dynamic_window(ann.start_s, ann.end_s, audio_dur)
    start_sample = int(win_start * sr)
    end_sample = int(win_end * sr)
    y_window = y[start_sample:end_sample]

    if len(y_window) < N_FFT:
        return None

    S = librosa.feature.melspectrogram(
        y=y_window, sr=sr,
        n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
        fmin=F_MIN, fmax=f_max_eff,
    )
    S_db = librosa.power_to_db(S, ref=np.max)

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=FIGURE_DPI)
        created_fig = True
    else:
        fig = ax.figure

    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel",
        fmin=F_MIN,fmax=f_max_eff,
        ax=ax,cmap=CMAP,
    )

    # Shift x-axis labels to absolute time
    xticks = ax.get_xticks()
    ax.set_xticklabels([f"{t + win_start:.2f}" for t in xticks])
    ax.set_xlabel("Time (s)")

    #Bounding box (in data coordinates)
    if show_bbox:
        bbox_x = ann.start_s - win_start# time offset
        bbox_w = ann.end_s - ann.start_s# width in seconds

        rect = patches.Rectangle(
            (bbox_x, ann.low_hz),
            bbox_w,
            ann.high_hz - ann.low_hz,
            linewidth=BBOX_LINEWIDTH,
            edgecolor=BBOX_COLOR,
            facecolor="none",
            linestyle="--",
        )
        ax.add_patch(rect)

        ax.text(
            bbox_x, ann.high_hz * 1.02,
            f"{ann.label}",
            color=BBOX_COLOR, fontsize=8, fontweight="bold",
            verticalalignment="bottom",
        )

    ax.set_title(
        f"{ann.audio_basename} │ {ann.label} │ "
        f"{ann.start_s:.3f}–{ann.end_s:.3f}s │ "
        f"{ann.low_hz/1000:.1f}–{ann.high_hz/1000:.1f} kHz",
        fontsize=9,
    )

    if created_fig:
        fig.colorbar(img, ax=ax, format="%+2.0f dB", pad=0.01)
        fig.tight_layout()
        return fig
    return None


#Render loop
n_show = min(MAX_ANNOTATIONS_TO_SHOW, len(annotations))
valid  = [a for a in annotations if a.audio_path is not None][:n_show]

print(f"Rendering {len(valid)} annotations ...\n")
for ann in valid:
    fig = render_annotation(ann, show_bbox=True)
    if fig:
        plt.show()
        plt.close(fig)

print(f"\n Rendered {len(valid)} spectrograms")

In [ ]:
# Cell 4 – Grid view: one row per species, up to 5 examples each
from collections import defaultdict

MAX_PER_SPECIES = 5

by_species: Dict[str, List[ParsedAnnotation]] = defaultdict(list)
for a in annotations:
    if a.audio_path is not None:
        by_species[a.label].append(a)

for species, anns in sorted(by_species.items()):
    subset = anns[:MAX_PER_SPECIES]
    n = len(subset)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5), dpi=FIGURE_DPI, squeeze=False)
    fig.suptitle(f"Species: {species} ({len(anns)} total)", fontsize=12, fontweight="bold")

    for i, ann in enumerate(subset):
        render_annotation(ann, ax=axes[0, i], show_bbox=True)
        axes[0, i].set_title(
            f"{ann.audio_basename}\n{ann.start_s:.2f}–{ann.end_s:.2f}s",
            fontsize=7,
        )

    fig.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()
    plt.close(fig)